In [1]:
import numpy as np
import openfhe_numpy as onp
from openfhe import *

In [2]:
mult_depth = 10

params = CCParamsCKKSRNS()
params.SetMultiplicativeDepth(mult_depth)
params.SetScalingModSize(59)
params.SetFirstModSize(60)
params.SetScalingTechnique(FIXEDAUTO)
params.SetKeySwitchTechnique(HYBRID)
params.SetSecretKeyDist(UNIFORM_TERNARY)

cc = GenCryptoContext(params)
cc.Enable(PKESchemeFeature.PKE)
cc.Enable(PKESchemeFeature.LEVELEDSHE)
cc.Enable(PKESchemeFeature.ADVANCEDSHE)

keys = cc.KeyGen()
cc.EvalMultKeyGen(keys.secretKey)
cc.EvalSumKeyGen(keys.secretKey)

batch_size = cc.GetRingDimension() // 2
print("\n****** CRYPTO PARAMETERS ******")
print(f"Total Slots: {batch_size}")
print("*******************************")


****** CRYPTO PARAMETERS ******
Total Slots: 32768
*******************************


### Testing out rotation of vector

In [125]:
vector1 = np.array([1.0, 2.0, 3.0, 4.0])
vector2 = np.array([5.0, 6.0, 7.0, 8.0])
vector3 = np.array([9.0, 10.0, 11.0, 12.0])

onp.gen_rotation_keys(keys.secretKey, [-7, -6, -5, -4, -3, -2, -1, 1, 2, 3, 4, 5, 6, 7])

enc_vec1 = onp.array(
    cc=cc,
    data=vector1,
    batch_size=batch_size,
    order=onp.ROW_MAJOR,
    mode="tile",
    fhe_type="C",
    public_key=keys.publicKey,
)

enc_vec2 = onp.array(
    cc=cc,
    data=vector2,
    batch_size=batch_size,
    order=onp.ROW_MAJOR,
    mode="tile",
    fhe_type="C",
    public_key=keys.publicKey,
)

enc_vec3 = onp.array(
    cc=cc,
    data=vector3,
    batch_size=batch_size,
    order=onp.ROW_MAJOR,
    mode="tile",
    fhe_type="C",
    public_key=keys.publicKey,
)

In [126]:
print(enc_vec1.decrypt(keys.secretKey, unpack_type="original"))
print(enc_vec2.decrypt(keys.secretKey, unpack_type="original"))
print(enc_vec3.decrypt(keys.secretKey, unpack_type="original"))

[1. 2. 3. 4.]
[5. 6. 7. 8.]
[ 9. 10. 11. 12.]


In [129]:
accum1 = enc_vec1.sum(axis=0)
accum2 = enc_vec2.sum(axis=0)
accum3 = enc_vec3.sum(axis=0)

In [130]:
print(accum1.decrypt(keys.secretKey, unpack_type="original"))
print(accum2.decrypt(keys.secretKey, unpack_type="original"))
print(accum3.decrypt(keys.secretKey, unpack_type="original"))

10.00000000000235
26.000000000001325
42.000000000003624


In [144]:
sum_vec = onp.array(
    cc=cc,
    data=np.zeros(len(vector1)),
    batch_size=batch_size,
    order=onp.ROW_MAJOR,
    mode="tile",
    fhe_type="C",
    public_key=keys.publicKey,
)

print(sum_vec.decrypt(keys.secretKey, unpack_type="original"))

[-1.09228837e-13 -3.33493910e-13  7.51515900e-14  5.87539948e-13]


In [145]:
selector_vector = np.zeros(len(vector))
selector_vector[0] = 1.0

pt_select = onp.array(
    cc=cc,
    data=selector_vector,
    batch_size=batch_size,
    order=onp.ROW_MAJOR,
    mode="tile",
    fhe_type="P",
    public_key=keys.publicKey,
)

In [146]:
sum_vec = sum_vec + onp.roll(pt_select * accum1, 0)

In [147]:
sum_vec = sum_vec + onp.roll(pt_select * accum2, 1)

In [148]:
sum_vec = sum_vec + onp.roll(pt_select * accum3, 2)

In [149]:
print(sum_vec.decrypt(keys.secretKey, unpack_type="original"))

[ 1.00000000e+01  2.60000000e+01  4.20000000e+01 -2.41549129e-15]


In [159]:
(pt_select * accum1).shape, (pt_select * accum1).original_shape

((4, 1), ())

In [153]:
vecs = [accum1, accum2, accum3]

In [156]:
def combine(vecs):
    
    sum_vec = onp.array(
        cc=cc,
        data=np.zeros(len(vector)),
        batch_size=batch_size,
        order=onp.ROW_MAJOR,
        mode="tile",
        fhe_type="C",
        public_key=keys.publicKey,
    )

    selector_vector = np.zeros(len(vector))
    selector_vector[0] = 1.0
    
    pt_select = onp.array(
        cc=cc,
        data=selector_vector,
        batch_size=batch_size,
        order=onp.ROW_MAJOR,
        mode="tile",
        fhe_type="P",
        public_key=keys.publicKey,
    )

    for i in range(len(vecs)):
        sum_vec = sum_vec + onp.roll(pt_select * vecs[i], i)

    return sum_vec

In [157]:
print(combine(vecs).decrypt(keys.secretKey, unpack_type="original"))

[1.00000000e+01 2.60000000e+01 4.20000000e+01 2.39719236e-13]


In [158]:
comb = combine(vecs)